# Deep Q-Learning (DQN) from Pixels – Mastering Atari Games

In this lab, you will implement the original Deep Q-Network (DQN) algorithm introduced by DeepMind in 2013. You will train an agent to play an Atari game (Pong or Breakout) directly from raw screen pixels.

Because we are processing high-dimensional image data and training a Convolutional Neural Network (CNN), this lab requires a GPU.

Hardware Check: Before starting, ensure your notebook environment is connected to a GPU.

In Google Colab: Runtime > Change runtime type > Select GPU.

Run the following cell to confirm GPU availability:

In [ ]:
!nvidia-smi

Thu Feb 19 15:53:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Part 1: Environment Setup and Exploration

Before building the brain, we need to understand the world our agent lives in. We will use Gymnasium (the modern version of OpenAI Gym) and its ALE (Arcade Learning Environment) wrapper.

Dependencies: Install and import gymnasium, torch, numpy, and matplotlib.

Initialize the Environment: Instantiate PongNoFrameskip-v4 or Breakout-v5.

Explore the Observation Space: Extract a single frame. Check its shape  and plot it. Notice the high dimensionality compared to the Taxi environment!

In [ ]:
# Si besoin dans Colab/Jupyter, décommentez la ligne suivante :
# !pip install "gymnasium[atari,accept-rom-license]" ale-py torch matplotlib

import random
from collections import deque, namedtuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ENV_ID = "PongNoFrameskip-v4"  # Alternative: "ALE/Breakout-v5"

# Environnement "brut" pour observer une frame RGB
raw_env = gym.make(ENV_ID, render_mode="rgb_array")
obs, _ = raw_env.reset()
print("Observation brute:", obs.shape, obs.dtype)

plt.figure(figsize=(6, 4))
plt.imshow(obs)
plt.title(f"Frame brute - {ENV_ID}")
plt.axis("off")
plt.show()

raw_env.close()


## Part 2: Image Preprocessing (Wrappers)

Feeding raw 210x160 RGB frames at 60 FPS into a neural network is computationally too expensive and inefficient for RL. We need to preprocess the data.

Your task is to implement or use existing Gym Wrappers to apply the following transformations:

Grayscale & Resize: Convert the image to grayscale and scale it down to 84x84 pixels.

Frame Skipping: The agent doesn't need to make a decision every single frame. Skip 4 frames at a time, repeating the last action.

Frame Stacking: A single static image contains no information about motion (velocity or direction of the ball). Stack the last 4 frames together to form a state of shape (4, 84, 84).

In [ ]:
from gymnasium.wrappers import AtariPreprocessing

try:
    # Gymnasium récent
    from gymnasium.wrappers import FrameStackObservation as FrameStackWrapper
except ImportError:
    # Compatibilité
    from gymnasium.wrappers import FrameStack as FrameStackWrapper


def make_env(env_id: str, seed: int = 0):
    """
    Préprocessing Atari:
    - frame_skip=4
    - grayscale + resize 84x84
    - stack de 4 frames
    """
    env = gym.make(env_id)
    env = AtariPreprocessing(
        env,
        frame_skip=4,
        screen_size=84,
        grayscale_obs=True,
        scale_obs=False,
    )
    env = FrameStackWrapper(env, 4)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    return env


env = make_env(ENV_ID, seed=42)
state, _ = env.reset()
state_np = np.array(state)

print("Observation prétraitée:", state_np.shape, state_np.dtype)
print("Nombre d'actions:", env.action_space.n)

# Visualisation des 4 frames stackées
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(state_np[i], cmap="gray")
    axes[i].set_title(f"Frame t-{3-i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()


## Part 3: The Deep Q-Network Architecture

PyTorch module DQN with the following architecture:

Input: (4, 84, 84) tensor.

Convolutional Blocks: 3 Conv2D layers (use the exact strides and kernel sizes from the DeepMind paper) followed by ReLU activations.

Fully Connected Layers: Flatten the output and pass it through a Linear layer.

Output Layer: A Linear layer with an output size equal to the number of possible actions (Q-values for each action).

In [ ]:
class DQN(nn.Module):
    def __init__(self, n_actions: int):
        super().__init__()
        # Architecture DeepMind (2013/2015)
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        # x attendu en [B, 4, 84, 84], uint8 ou float
        if x.dtype != torch.float32:
            x = x.float()
        x = x / 255.0
        x = self.conv(x)
        return self.head(x)


n_actions = env.action_space.n
policy_net = DQN(n_actions).to(device)
target_net = DQN(n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

print(policy_net)


## Part 4: Core DQN Components

To make DQN converge, we discussed two crucial innovations. You need to implement them:
- Experience Replay Buffer: A circular buffer that stores transitions (state, action, reward, next_state, done).Write a method to push experiences and sample a randomized batch to break the correlation between consecutive frames
- Epsilon-Greedy Strategy: Implement an action selection function with an exploration rate ($\epsilon$) that decays over time.

In [ ]:
Transition = namedtuple("Transition", ("state", "action", "reward", "next_state", "done"))


class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)


BATCH_SIZE = 32
GAMMA = 0.99
LR = 1e-4
BUFFER_SIZE = 100_000

EPS_START = 1.0
EPS_END = 0.1
EPS_DECAY = 200_000  # décroissance sur nb de steps

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
replay_buffer = ReplayBuffer(BUFFER_SIZE)

steps_done = 0


def epsilon_by_step(step):
    ratio = min(1.0, step / EPS_DECAY)
    return EPS_START + ratio * (EPS_END - EPS_START)


def select_action(state_np):
    global steps_done
    eps = epsilon_by_step(steps_done)
    steps_done += 1

    if random.random() < eps:
        return env.action_space.sample(), eps

    with torch.no_grad():
        state_t = torch.as_tensor(state_np, device=device).unsqueeze(0)
        q_values = policy_net(state_t)
        action = int(q_values.argmax(dim=1).item())
    return action, eps


def optimize_model():
    if len(replay_buffer) < BATCH_SIZE:
        return None

    transitions = replay_buffer.sample(BATCH_SIZE)

    state_batch = torch.as_tensor(np.array(transitions.state), device=device)
    action_batch = torch.as_tensor(transitions.action, device=device).long().unsqueeze(1)
    reward_batch = torch.as_tensor(transitions.reward, device=device).float()
    next_state_batch = torch.as_tensor(np.array(transitions.next_state), device=device)
    done_batch = torch.as_tensor(transitions.done, device=device).float()

    # Q(s,a)
    q_values = policy_net(state_batch).gather(1, action_batch).squeeze(1)

    # max_a' Q_target(s',a')
    with torch.no_grad():
        next_q_values = target_net(next_state_batch).max(1).values
        target = reward_batch + GAMMA * next_q_values * (1.0 - done_batch)

    loss = nn.functional.smooth_l1_loss(q_values, target)

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(policy_net.parameters(), 10.0)
    optimizer.step()

    return float(loss.item())


## Part 5: The Training Loop

Implement the main training loop


In [ ]:
NUM_EPISODES = 50          # mettez 300+ pour un vrai apprentissage
TARGET_UPDATE_FREQ = 1_000 # en steps
MAX_STEPS_PER_EPISODE = 10_000

episode_rewards = []
losses = []

env = make_env(ENV_ID, seed=42)
state, _ = env.reset()
state = np.array(state)

for episode in range(1, NUM_EPISODES + 1):
    state, _ = env.reset()
    state = np.array(state)
    done = False
    truncated = False
    total_reward = 0.0

    for t in range(MAX_STEPS_PER_EPISODE):
        action, eps = select_action(state)
        next_state, reward, done, truncated, _ = env.step(action)
        next_state = np.array(next_state)
        terminal = done or truncated

        replay_buffer.push(state, action, reward, next_state, terminal)
        state = next_state
        total_reward += reward

        loss = optimize_model()
        if loss is not None:
            losses.append(loss)

        if steps_done % TARGET_UPDATE_FREQ == 0:
            target_net.load_state_dict(policy_net.state_dict())

        if terminal:
            break

    episode_rewards.append(total_reward)

    if episode % 5 == 0:
        avg_last = np.mean(episode_rewards[-5:])
        print(
            f"Episode {episode:03d} | Reward: {total_reward:6.1f} | "
            f"Moyenne(5): {avg_last:6.2f} | Epsilon: {eps:.3f}"
        )

env.close()
print("Entraînement terminé.")


## Part 6: Evaluation and Visualization

Once your agent has trained display its performance ! (rewards vs. episodes)

In [ ]:
# Courbe des rewards
plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, label="Reward / épisode")
if len(episode_rewards) >= 10:
    moving_avg = np.convolve(episode_rewards, np.ones(10)/10, mode="valid")
    plt.plot(range(9, len(episode_rewards)), moving_avg, label="Moyenne glissante (10)")
plt.xlabel("Épisode")
plt.ylabel("Reward")
plt.title("Performance DQN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Évaluation gloutonne (epsilon=0)
eval_env = make_env(ENV_ID, seed=123)
state, _ = eval_env.reset()
state = np.array(state)

done = False
truncated = False
eval_reward = 0.0

while not (done or truncated):
    with torch.no_grad():
        state_t = torch.as_tensor(state, device=device).unsqueeze(0)
        action = int(policy_net(state_t).argmax(dim=1).item())
    state, reward, done, truncated, _ = eval_env.step(action)
    state = np.array(state)
    eval_reward += reward

eval_env.close()
print(f"Reward d'évaluation (politique gloutonne): {eval_reward:.1f}")
